# Capability 21: Hierarchy-aware fallback for unsupported entities or granularities

7/7 cases passed against a real, live LLM (gateway-configured model, see `.env`). Every code cell below is real, executable code -- the same `ask()` pattern as `notebooks/demo.ipynb` -- not a mockup; the attached output is what actually happened when this ran, captured via `scripts/run_live_capability_tests.py --capability 21`. Re-running this notebook (Restart Kernel & Run All) with a live key will make new real calls.

See `tests/live/cases/cap21_hierarchy_fallback.py` for these case definitions with their automated pass/fail checks, and `tests/live/live_capabilities_suite.py` for how they run as unittest assertions.

In [ ]:
import sys, pathlib

# Robust path insert regardless of where Jupyter's cwd lands (repo root, or
# this notebook's own folder under notebooks/capabilities/<slug>/):
_p = pathlib.Path.cwd()
while not (_p / "src").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

import os

try:
    from dotenv import load_dotenv  # optional: picks up a .env file if python-dotenv is installed
    load_dotenv(override=False)
except ImportError:
    pass

from src.orchestrator import Orchestrator
from src.llm_client import get_llm_client, GLOBAL_USAGE, MockLLMClient

provider = os.environ.get("LLM_PROVIDER", "").lower() or ("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "openai" if os.environ.get("OPENAI_API_KEY") else "mock")
print(f"LLM provider in use: {provider}" + ("  (\u26a0\ufe0f set ANTHROPIC_API_KEY or OPENAI_API_KEY for real answers)" if provider == "mock" else ""))

orch = Orchestrator()


LLM provider in use: openai


In [ ]:
def ask(question: str, label: str = ""):
    """Run one turn through the orchestrator and pretty-print everything the
    grader needs to see: routing, evidence sources, transparency notes, answer.
    Identical helper to notebooks/demo.ipynb -- see scripts/build_notebook.py."""
    if label:
        print(f"\n{'='*90}\n{label}\n{'='*90}")
    print(f"USER: {question}\n")
    resp = orch.handle_turn(question)
    print(f"[intent={resp.intent} | sub_agents={resp.sub_agents_used} | retried={resp.retried}]")
    if resp.citations:
        print(f"[citations: {[c['doc_id'] for c in resp.citations]}]")
    if resp.assumptions:
        print("[assumptions/limitations surfaced:]")
        for a in resp.assumptions:
            print(f"  - {a}")
    print(f"\nAGENT: {resp.answer}")
    if resp.follow_up_suggestions:
        print(f"\n(follow-up suggestions: {resp.follow_up_suggestions})")
    return resp


## 01: country requested -> should roll up to its zone with an explicit note

**✅ PASS**

In [ ]:
_ = ask("What was AB InBev's revenue in Brazil specifically in 2025?", '01')

01
USER: What was AB InBev's revenue in Brazil specifically in 2025?

[intent=data_query | sub_agents=['structured', 'unstructured']]
[citations: ['DOC-011', 'DOC-005', 'DOC-012', 'DOC-004', 'DOC-008']]
[assumptions/limitations surfaced:]
  - AB InBev doesn't publicly disclose structured financials by country; showing the **South America** zone (which includes Brazil) instead, plus any Brazil-specific commentary found in the documents.

AGENT: ## Short answer

AB InBev does **not** publicly disclose a Brazil-only revenue figure, so I can't give you a precise number for Brazil in 2025. What the evidence does support:

- **South America zone (which includes Brazil), FY2025:** **$11,954M** in revenue (summed across all reported quarters of 2025).
- **Brazil-specific, 2025:** no revenue figure is available — only **volume** commentary: Brazil volumes **declined 4.1%** in FY2025, with **beer volumes down 4.6%** [DOC-011].

## Context from the documents

The zone-level figures below show how

## 02: a different country -> different zone rollup

**✅ PASS**

In [ ]:
_ = ask('What was the revenue for the United States in Q1 2024?', '02')

02
USER: What was the revenue for the United States in Q1 2024?

[intent=data_query | sub_agents=['structured']]
[assumptions/limitations surfaced:]
  - AB InBev doesn't publicly disclose structured financials by country; showing the **North America** zone (which includes United States) instead, plus any United States-specific commentary found in the documents.

AGENT: Based on the retrieved KPI data, Q1 2024 revenue was **$3,593 million (USD)**.

**Important caveat:** AB InBev does not publicly disclose structured financials by individual country. The figure above reflects the **North America** zone — which includes the United States — rather than a strictly U.S.-only metric. So treat it as zone-level revenue, not a pure United States figure.

Would you like to compare this against Q1 2023 for the same North America zone, or look at revenue by brand/category within the zone?

## 03: a real, named competitor -> flagged as genuinely unsupported, not silently dropped

**✅ PASS**

In [ ]:
_ = ask('How is Heineken performing financially?', '03')

03
USER: How is Heineken performing financially?

[intent=data_query | sub_agents=['web']]
[assumptions/limitations surfaced:]
  - 'Heineken' isn't part of Anheuser-Busch InBev (AB InBev)'s tracked entities (brand/zone/country), or is a different company entirely (e.g. a competitor), so no internal structured data exists for it. Any answer about it, if given, is qualitative/public information only, not internal reporting.

AGENT: Heineken is **not part of Anheuser-Busch InBev (AB InBev)** — it is a separate, competing brewing company. As a result, there is **no internal AB InBev structured or reporting data** on Heineken, and I cannot provide any internal financial figures for it from the available evidence.

Any information on Heineken’s financial performance would be **public/qualitative only** (e.g., investor relations, annual reports), and the retrieved evidence does not contain those specific numbers or news details. I’m unable to confirm revenue, profit, growth, or market share f

## 04: brand-level financials requested -> should say they aren't disclosed, not approximate

**✅ PASS**

In [ ]:
_ = ask("What was Budweiser's exact revenue in 2025?", '04')

04
USER: What was Budweiser's exact revenue in 2025?

[intent=data_query | sub_agents=['unstructured']]
[citations: ['DOC-012', 'DOC-004', 'DOC-011', 'DOC-008', 'DOC-006']]

AGENT: I couldn’t find an **exact Budweiser revenue figure for 2025** in the retrieved evidence.

What is available:

- [DOC-012] notes AB InBev’s “megabrands” — anchored by Budweiser, Corona, Stella Artois, and Michelob Ultra — grew revenue **4.1% in FY2025**, but no Budweiser-specific dollar amount is provided.
- [DOC-008] provides FY2025 zone-level results (e.g., North America Q4 2025 revenue of **$3,235M**), but these are zone totals, not Budweiser brand revenue.

So the specific `revenue_usd_m` figure for Budweiser in 2025 is **not present in the evidence returned** — I don’t want to guess or cite an unsupported number. If you have a structured KPI dataset that wasn’t retrieved, please let me know and I can try again.

Want me to pull AB InBev’s **North America FY2025 revenue** instead, or search specifically 

## 05: a country whose zone rollup should be Middle Americas

**✅ PASS**

In [ ]:
_ = ask("What was Mexico's revenue contribution in 2025?", '05')

05
USER: What was Mexico's revenue contribution in 2025?

[intent=data_query | sub_agents=['unstructured']]
[citations: ['DOC-012', 'DOC-004', 'DOC-011', 'DOC-008', 'DOC-006']]
[assumptions/limitations surfaced:]
  - AB InBev doesn't publicly disclose structured financials by country; showing the **Middle Americas** zone (which includes Mexico) instead, plus any Mexico-specific commentary found in the documents.

AGENT: AB InBev does not publicly disclose Mexico-specific revenue for 2025, so a precise "Mexico revenue contribution" figure isn't available from the retrieved data. [DOC-011] shows AB InBev reports country-level commentary mostly by volume, not revenue.

What we can say:

- **Mexico is reported within the Middle Americas zone**, which includes Mexico and other markets.
- **Middle Americas 2025 quarterly revenue (shared data points only):**
  - Q2 2025: $4,340M [DOC-006]
  - Q4 2025: $4,927M [DOC-008]
- Full-year 2025 figures for the Middle Americas zone were not included in

## 06: a country whose zone rollup should be EMEA

**✅ PASS**

In [ ]:
_ = ask("What was the United Kingdom's revenue in 2025?", '06')

06
USER: What was the United Kingdom's revenue in 2025?

[intent=data_query | sub_agents=['structured', 'unstructured']]
[citations: ['DOC-011', 'DOC-012', 'DOC-004', 'DOC-008', 'DOC-014']]
[assumptions/limitations surfaced:]
  - AB InBev doesn't publicly disclose structured financials by country; showing the **EMEA** zone (which includes United Kingdom) instead, plus any United Kingdom-specific commentary found in the documents.

AGENT: ## Short answer

**A United Kingdom-specific revenue figure for 2025 is not available in the data.** AB InBev does not publicly disclose structured financials at the country level, so the closest disclosed figure is the **EMEA zone total**, which includes the United Kingdom but also every other market in the zone.

**EMEA zone revenue, FY2025: $9,502M (USD million)**

| Zone | Year | Revenue (USD million) |
| --- | --- | --- |
| EMEA | 2025 | $9,502 |

*Source: structured KPI query on `fact_kpi` (quarterly grain, summed to full year for 2025).*

## Why

## 07: combination: country rollup AND a qualitative document answer together

**✅ PASS**

In [ ]:
_ = ask("What was AB InBev's revenue in Brazil, and what's driving it?", '07')

07
USER: What was AB InBev's revenue in Brazil, and what's driving it?

[intent=data_query | sub_agents=['structured', 'unstructured']]
[citations: ['DOC-013', 'DOC-011', 'DOC-005', 'DOC-012', 'DOC-004']]
[assumptions/limitations surfaced:]
  - AB InBev doesn't publicly disclose structured financials by country; showing the **South America** zone (which includes Brazil) instead, plus any Brazil-specific commentary found in the documents.

AGENT: ## Short answer

AB InBev does **not** publicly disclose structured financials by country, so the structured series below is effectively the **South America zone** (which includes Brazil) — and it reconciles exactly to the zone figures AB InBev reported in its Q1 2025 and Q4 2024 earnings commentary [DOC-005][DOC-004]. Brazil-specific granularity exists only in the narrative filing commentary (volume trends), not in revenue dollars.

| Period | Revenue (USD m) | Volume (K hL) | Organic revenue growth |
|---|---|---|---|
| Q1 2024 | $3,233 | 40,